# Simple Agent with Strands SDK

This notebook demonstrates how to create a simple AI agent using the Strands Agents SDK with Amazon Bedrock and Neo4j.

The agent will:
- Connect to your Neo4j database
- Use a tool to retrieve the graph schema
- Answer questions about the database structure

---

## Setup

Import required modules and configure the environment.

In [ ]:
import sys
sys.path.insert(0, '..')

from strands import Agent, tool
from strands.models import BedrockModel
from neo4j_graphrag.schema import get_schema

from config import get_neo4j_driver, aws_config

## Connect to Neo4j

Create a connection to your Neo4j database.

In [ ]:
driver = get_neo4j_driver()
driver.verify_connectivity()
print("Connected to Neo4j successfully!")

## Define Tools

In Strands, tools are Python functions decorated with `@tool`. The agent uses the function name and docstring to decide when to execute the tool.

> **Important:** Write clear, descriptive docstrings - the agent relies on them to understand when to use each tool.

In [ ]:
@tool
def get_graph_schema() -> str:
    """
    Get the schema of the graph database including node labels, 
    relationship types, and properties.
    
    Use this tool when the user asks about:
    - What data is in the database
    - What types of nodes or relationships exist
    - The structure of the graph
    - What questions can be answered
    
    Returns:
        A string describing the database schema.
    """
    return get_schema(driver)

print("Tool defined: get_graph_schema")

## Configure Bedrock Model

Create a Bedrock model configuration for Claude.

In [ ]:
bedrock_model = BedrockModel(
    model_id=aws_config.bedrock_model_id,
    region_name=aws_config.region,
    temperature=0.3,
)

print(f"Model configured: {aws_config.bedrock_model_id}")

## Create the Agent

Create an agent with the Bedrock model and the schema tool.

In [ ]:
agent = Agent(
    model=bedrock_model,
    system_prompt="""
    You are a helpful assistant that answers questions about a Neo4j graph database 
    containing SEC 10-K financial filings.
    
    You have access to a tool that can retrieve the database schema. Use it when 
    users ask about the structure of the data or what questions can be answered.
    
    Be concise and informative in your responses.
    """,
    tools=[get_graph_schema]
)

print("Agent created successfully!")

## Run the Agent

Ask the agent a question about the database schema.

In [ ]:
query = "Summarize the schema of the graph database."

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

**What happened:**
1. The agent received the user's query
2. It recognized the need for schema information
3. It called the `get_graph_schema` tool
4. It used the tool's output to formulate a response

## Try Different Questions

Experiment with different questions about the database.

In [ ]:
query = "How are Products related to other entities in the database?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "What questions can I answer using this graph database?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

In [ ]:
query = "How does the graph model relate financial documents to risk factors?"

print(f"User: {query}\n")
print("Assistant:")

response = agent(query)
print(response)

## Summary

In this notebook, you learned:

1. **@tool decorator** - How to create tools for Strands agents
2. **BedrockModel** - Configuring Amazon Bedrock as the model provider
3. **Agent creation** - Building an agent with tools and a system prompt
4. **Tool invocation** - How agents automatically select and use tools

The agent used the schema tool to answer questions about the database structure. In the next notebook, you'll add a vector search tool to answer questions about document content.

---

**Next:** [Vector Graph Agent](02_vector_graph_agent.ipynb)

In [ ]:
# Cleanup
driver.close()
print("Connection closed.")